Installing Phase:

In [ ]:
!pip install opencv-python
!pip install matplotlib
!pip install numpy
!pip install scikit-learn
!pip install pandas
!pip install tqdm
!pip install scikit-image
!pip install scipy
!pip install ace_tools

Calling the Libraries:

In [ ]:
from skimage.feature import local_binary_pattern
from skimage.color import rgb2gray
from skimage import exposure
from skimage.io import imread
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import normalize
import cv2
import glob
import os

Finding Height and Width of an Image:

In [ ]:
import cv2
import os

# Example sample image path from session 1
sample_image_path = '/content/drive/MyDrive/Datasets/FV_USM/Published_database_FV-USM_Dec2013/Published_database_FV-USM_Dec2013/1st_session/extractedvein/vein001_1/01.jpg'

# Load the image in grayscale
img = cv2.imread(sample_image_path, cv2.IMREAD_GRAYSCALE)

# Check if image was loaded successfully
if img is None:
    print("Image could not be loaded. Check the path.")
else:
    # Print its shape
    print("Image shape:", img.shape)

    # Print height and width
    height, width = img.shape
    print("Height:", height)
    print("Width:", width)

Train:

In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from skimage.feature import local_binary_pattern
from sklearn.preprocessing import normalize
import pandas as pd

# ----------------------------
# CONFIGURATION
# ----------------------------
base_path_sess1 = '/content/drive/MyDrive/Datasets/FV_USM/Published_database_FV-USM_Dec2013/Published_database_FV-USM_Dec2013/1st_session/extractedvein'
base_path_sess2 = '/content/drive/MyDrive/Datasets/FV_USM/Published_database_FV-USM_Dec2013/Published_database_FV-USM_Dec2013/2nd_session/extractedvein'

NUM_SUBJECTS = 123
NUM_FINGERS = 4
IMAGES_PER_SESSION = 3
IMAGE_SIZE = (100, 300)
PATCH_SIZE = 10
STRIDE = 10
# LBP_CONFIGS = [(1, 16), (1, 8), (2, 8)]  # (radius, points)
LBP_CONFIGS = [(2, 8)]

# ----------------------------
# FUNCTION: RIU2 Mapping
# ----------------------------
def get_riu2_mapping(P):
    table = np.zeros(2 ** P, dtype=np.uint8)
    for i in range(2 ** P):
        binary = [(i >> j) & 1 for j in range(P)]
        rotations = [binary[n:] + binary[:n] for n in range(P)]
        min_rotation = min(rotations)
        extended = min_rotation + [min_rotation[0]]
        transitions = sum(extended[j] != extended[j + 1] for j in range(P))
        table[i] = sum(min_rotation) if transitions <= 2 else P + 1
    return table

# ----------------------------
# FUNCTION: Extract LBP Histogram
# ----------------------------
def extract_lbp_histogram(block, P, R, riu2_map):
    lbp = local_binary_pattern(block, P, R, method='ror').astype(np.uint16)
    lbp_mapped = riu2_map[lbp]
    hist, _ = np.histogram(
        lbp_mapped.ravel(),
        bins=np.arange(0, P + 3),
        density=True
    )
    return hist

# ----------------------------
# PRECOMPUTE MAPPINGS
# ----------------------------
mapping_dict = {P: get_riu2_mapping(P) for _, P in LBP_CONFIGS}

# ----------------------------
# FEATURE EXTRACTION (Both Sessions, Strategy 2)
# ----------------------------
all_features = []
all_labels = []

print("🚀 Starting feature extraction from both sessions (Strategy 2 - separate fingers)...")

for subject_id in tqdm(range(1, NUM_SUBJECTS + 1)):
    for finger_id in range(1, NUM_FINGERS + 1):
        folder_name = f"vein{subject_id:03d}_{finger_id}"

        for base_path in [base_path_sess1, base_path_sess2]:
            session = 's1' if '1st_session' in base_path else 's2'
            folder_path = os.path.join(base_path, folder_name)

            for i in range(1, IMAGES_PER_SESSION + 1):
                img_path = os.path.join(folder_path, f"{i:02d}.jpg")
                print(f"📷 Processing {img_path} → Subject: {subject_id}, Finger: {finger_id}, Session: {session}, Image: {i}")

                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                if img is None:
                    print(f"❌ Skipped missing: {img_path}")
                    continue

                resized = cv2.resize(img, IMAGE_SIZE)
                denoised = cv2.fastNlMeansDenoising(resized, h=10)
                equalized = cv2.equalizeHist(denoised).astype(np.float64) / 255.0
                normalized = (equalized - np.mean(equalized)) / np.std(equalized)

                fused_vector = []
                for y in range(0, IMAGE_SIZE[1] - PATCH_SIZE + 1, STRIDE):
                    for x in range(0, IMAGE_SIZE[0] - PATCH_SIZE + 1, STRIDE):
                        block = normalized[y:y + PATCH_SIZE, x:x + PATCH_SIZE]
                        combined_hist = []
                        for R, P in LBP_CONFIGS:
                            hist = extract_lbp_histogram(block, P, R, mapping_dict[P])
                            combined_hist.extend(hist)
                        fused_vector.extend(combined_hist)

                if fused_vector:
                    all_features.append(fused_vector)
                    # Clean label format for matching: 074_img2_ss1
                    label = f"{subject_id:03d}_img{i}_ss{session[-1]}"
                    all_labels.append(label)

# ----------------------------
# NORMALIZATION & SUMMARY
# ----------------------------
if all_features:
    all_features = np.array(all_features, dtype=np.float32)
    all_features = normalize(all_features, norm='l2')
    all_labels = np.array(all_labels)

    df = pd.DataFrame({
        "Label": all_labels,
        "Feature Length": [len(vec) for vec in all_features]
    })

    print("\n📊 Feature Summary (First 5 Samples):")
    print(df.head())

    print("\n✅ Feature extraction complete!")
    print("🔢 Feature matrix shape:", all_features.shape)
    print("🟢 Sample labels:", all_labels[:5])
else:
    print("⚠️ No features extracted. Please check image paths or data integrity.")


Test:

In [ ]:
# ----------------------------
# TEST FEATURE EXTRACTION (Images 05 & 06 – Strategy 2 – Protocol 1)
# ----------------------------
test_lbp_features = []
test_labels = []
test_paths = []

print("🧪 Extracting test features from images 05 & 06 (Strategy 2 - Protocol 1)...")

for subject_id in tqdm(range(1, NUM_SUBJECTS + 1), desc="LBP Test Data (S2-P1)"):
    for finger_id in range(1, NUM_FINGERS + 1):
        folder_name = f"vein{subject_id:03d}_{finger_id}"

        for base_path in [base_path_sess1, base_path_sess2]:
            session = 's1' if '1st_session' in base_path else 's2'
            folder_path = os.path.join(base_path, folder_name)

            for img_idx in [4, 5, 6]:  # Testing images (Protocol 2)
                img_path = os.path.join(folder_path, f"{img_idx:02d}.jpg")
                print(f"📸 Loading: {img_path}")

                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                if img is None:
                    print(f"❌ Missing image: {img_path}")
                    continue

                resized = cv2.resize(img, IMAGE_SIZE)
                denoised = cv2.fastNlMeansDenoising(resized, h=10)
                equalized = cv2.equalizeHist(denoised).astype(np.float64) / 255.0
                normalized = (equalized - np.mean(equalized)) / (np.std(equalized) + 1e-8)

                fused_vector = []
                for y in range(0, IMAGE_SIZE[1] - PATCH_SIZE + 1, STRIDE):
                    for x in range(0, IMAGE_SIZE[0] - PATCH_SIZE + 1, STRIDE):
                        block = normalized[y:y + PATCH_SIZE, x:x + PATCH_SIZE]
                        combined_hist = []
                        for R, P in LBP_CONFIGS:
                            hist = extract_lbp_histogram(block, P, R, mapping_dict[P])
                            combined_hist.extend(hist)
                        fused_vector.extend(combined_hist)

                if fused_vector:
                    test_lbp_features.append(fused_vector)
                    # Consistent label format with training set
                    label = f"{subject_id:03d}_img{img_idx}_ss{session[-1]}"
                    test_labels.append(label)
                    test_paths.append(img_path)

# ----------------------------
# NORMALIZE TEST FEATURES
# ----------------------------
if test_lbp_features:
    test_lbp_features = np.array(test_lbp_features, dtype=np.float32)
    test_lbp_features = normalize(test_lbp_features, norm='l2')
    test_labels = np.array(test_labels)

    print(f"\n✅ Test feature extraction complete. Shape: {test_lbp_features.shape}")
    print(f"🟢 First few test labels: {test_labels[:5]}")
else:
    print("⚠️ No test features extracted. Please check image paths or image existence.")


Benchmarking:

In [ ]:
# Helper function
train_lbp_features = all_features
train_labels = all_labels

def extract_subject_and_session(label):
    parts = label.split('_')
    subject = parts[0]  # e.g., "003"
    session = parts[-1]  # e.g., "s1"
    return subject, session

correct_matches = 0
total_tests = len(test_lbp_features)

print("\n🔍 Classifying test data using Manhattan distance (Match: Subject ID + Session)...\n")

for i in range(total_tests):
    test_vec = test_lbp_features[i]
    true_label = test_labels[i]

    distances = np.sum(np.abs(train_lbp_features - test_vec), axis=1)
    min_index = np.argmin(distances)
    predicted_label = train_labels[min_index]

    true_subj, true_sess = extract_subject_and_session(true_label)
    pred_subj, pred_sess = extract_subject_and_session(predicted_label)

    if true_subj == pred_subj and true_sess == pred_sess:
        correct_matches += 1
        match_symbol = "✅"
    else:
        match_symbol = "❌"

    print(f"Test sample {i+1}: Predicted = {predicted_label}, Actual = {true_label} {match_symbol}")

accuracy = (correct_matches / total_tests) * 100
print("\n📊 Final Results")
print(f"✅ Correct matches: {correct_matches} / {total_tests}")
print(f"🎯 Recognition Accuracy: {accuracy:.2f}%")


Session Sensitive R5

In [ ]:
import numpy as np
from collections import defaultdict

# Define the ranks you want to evaluate
ranks = [1, 5]
rank_correct = defaultdict(int)
total_tests = len(test_labels)

print("📊 Calculating Session-Sensitive CMC (Rank-1 & Rank-5)...")

for i in range(total_tests):
    proj_test = test_lbp_features[i]
    test_label = test_labels[i]

    # Extract subject and session from test label
    test_parts = test_label.split('_')
    test_subject = test_parts[0]
    test_session = test_parts[-1]
    test_id = f"{test_subject}_{test_session}"

    # Calculate Manhattan distances to all training samples
    distances = np.sum(np.abs(train_lbp_features - proj_test), axis=1)
    sorted_indices = np.argsort(distances)

    matched = False
    for r in range(1, max(ranks) + 1):
        candidate_label = train_labels[sorted_indices[r - 1]]
        parts = candidate_label.split('_')
        candidate_subject = parts[0]
        candidate_session = parts[-1]
        candidate_id = f"{candidate_subject}_{candidate_session}"

        if candidate_id == test_id and not matched:
            for k in ranks:
                if r <= k:
                    rank_correct[k] += 1
            matched = True

# Final results
for k in ranks:
    accuracy = (rank_correct[k] / total_tests) * 100
    print(f"🎯 Rank-{k} Accuracy (Subject + Session): {accuracy:.2f}%")


Session Sensitive CMC

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# === CONFIGURATION ===
max_rank = 100
rank_correct = np.zeros(max_rank)
total_tests = len(test_lbp_features)

print("📊 Calculating Session-Sensitive CMC Curve (Rank-1 to Rank-100)...")

for i in range(total_tests):
    proj_test = test_lbp_features[i]
    test_parts = test_labels[i].split('_')
    true_subject = test_parts[0]
    true_session = test_parts[-1]
    true_id = f"{true_subject}_{true_session}"  # Matching on subject + session

    # Compute Manhattan distances to all training samples
    distances = np.sum(np.abs(train_lbp_features - proj_test), axis=1)
    sorted_indices = np.argsort(distances)

    # Find the first correct match
    for r in range(max_rank):
        candidate_label = train_labels[sorted_indices[r]]
        cand_parts = candidate_label.split('_')
        cand_subject = cand_parts[0]
        cand_session = cand_parts[-1]
        candidate_id = f"{cand_subject}_{cand_session}"

        if candidate_id == true_id:
            rank_correct[r:] += 1  # All ranks >= r count this match
            break

# === Normalize to percentage
cmc_curve = (rank_correct / total_tests) * 100

# === Plotting the CMC Curve
plt.figure(figsize=(10, 6))
plt.plot(np.arange(1, max_rank + 1), cmc_curve, label="Session-Sensitive CMC", linewidth=2)
plt.xlabel("Rank")
plt.ylabel("Identification Accuracy (%)")
plt.title("Session-Sensitive CMC Curve — LBP$_{\\mathrm{RIU2}}$((8,1), (16,1), (8,2)) (Strategy 2, Protocol 2)")
plt.grid(True)
plt.xticks(np.arange(0, max_rank + 1, 10))
plt.legend()
plt.tight_layout()
plt.show()

# === Print Key Rank Accuracies
print(f"🎯 Rank-1 Accuracy   : {cmc_curve[0]:.2f}%")
print(f"🎯 Rank-5 Accuracy   : {cmc_curve[4]:.2f}%")
print(f"🎯 Rank-10 Accuracy  : {cmc_curve[9]:.2f}%")
print(f"🎯 Rank-100 Accuracy : {cmc_curve[99]:.2f}%")


Session Sensitive Precision, Recall, F1, Accuracy

In [ ]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

# === Initialize lists
all_scores = []
all_labels = []

# === Pairwise score computation (Session-Sensitive)
for test_idx in range(len(test_lbp_features)):
    test_vec = test_lbp_features[test_idx]
    test_label = test_labels[test_idx]
    test_parts = test_label.split('_')
    test_subject = test_parts[0]
    test_session = test_parts[-1]
    test_id = f"{test_subject}_{test_session}"

    for train_idx in range(len(train_lbp_features)):
        train_vec = train_lbp_features[train_idx]
        train_label = train_labels[train_idx]
        train_parts = train_label.split('_')
        train_subject = train_parts[0]
        train_session = train_parts[-1]
        train_id = f"{train_subject}_{train_session}"

        # ✅ Negative Manhattan distance (higher score = more similar)
        score = -np.sum(np.abs(test_vec - train_vec))
        all_scores.append(score)

        # ✅ Label as genuine (1) if subject and session match
        is_genuine = int(test_id == train_id)
        all_labels.append(is_genuine)

# === Normalize similarity scores to [0, 1]
scores = np.array(all_scores)
labels = np.array(all_labels)
scores = (scores - scores.min()) / (scores.max() - scores.min())

# === Threshold Sweeping to Maximize F1 Score
best_f1 = best_thresh = best_prec = best_rec = 0

for t in np.linspace(0, 1, 1000):
    preds = (scores >= t).astype(int)
    precision = precision_score(labels, preds, zero_division=0)
    recall = recall_score(labels, preds, zero_division=0)
    f1 = f1_score(labels, preds, zero_division=0)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t
        best_prec = precision
        best_rec = recall

# === Final classification at best threshold
final_preds = (scores >= best_thresh).astype(int)
accuracy = accuracy_score(labels, final_preds)

# === Output Summary for Paper/Table
print("🔍 Summary (Session-Sensitive Verification): LBP((8,1),(16,1),(8,2))")
print(f"📍 Optimal Threshold  : {best_thresh:.3f}")
print(f"✔️ Accuracy           : {accuracy * 100:.2f}%")
print(f"✔️ Precision (PR)     : {best_prec * 100:.2f}%")
print(f"✔️ Recall (RC)        : {best_rec * 100:.2f}%")
print(f"✔️ F1 Score (F1)      : {best_f1 * 100:.2f}%")
